In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io

In [3]:
file_path = "../../01_Raw_Data/HR_Employee_Attrition.csv"

df = pd.read_csv(file_path)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../../01_Raw_Data/HR_Employee_Attrition.csv'

Create a Working Copy

In [ ]:
file_path = "../../01_Raw_Data/HR_Employee_Attrition.csv"

df = pd.read_csv(file_path)

df.head()

initial_rows = clean_df.shape[0]
initial_columns = clean_df.shape[1]

print("Initial Rows:", initial_rows)
print("Initial Columns:", initial_columns)

Check Missing Values Again

In [ ]:
missing_summary = clean_df.isnull().sum()

missing_summary[missing_summary > 0]

,0


In [ ]:
total_missing = clean_df.isnull().sum().sum()

print("Total Missing Values:", total_missing)

Total Missing Values: 0


Check Duplicate Rows

In [ ]:
duplicate_count = clean_df.duplicated().sum()

print("Duplicate Rows:", duplicate_count)

Duplicate Rows: 0


Validate EmployeeNumber

In [ ]:
employee_id_duplicates = clean_df["EmployeeNumber"].duplicated().sum()

print("Duplicate Employee IDs:", employee_id_duplicates)

Duplicate Employee IDs: 0


In [ ]:
print("Unique Employee IDs:",
      clean_df["EmployeeNumber"].nunique())

Unique Employee IDs: 1470


Check Data Types

In [ ]:
clean_df.dtypes

,0
Age,int64
Attrition,object
BusinessTravel,object
DailyRate,int64
Department,object
DistanceFromHome,int64
Education,int64
EducationField,object
EmployeeCount,int64
EmployeeNumber,int64


Convert Categorical Columns to category

In [ ]:
categorical_columns = clean_df.select_dtypes(include='object').columns
for column in categorical_columns:
    clean_df[column] = clean_df[column].astype("category")

In [ ]:
clean_df[categorical_columns].dtypes

,0
Attrition,category
BusinessTravel,category
Department,category
EducationField,category
Gender,category
JobRole,category
MaritalStatus,category
Over18,category
OverTime,category


Validate Categorical Values

In [ ]:
for column in categorical_columns:
    print(f"\n{column}:")
    print(clean_df[column].unique())


Attrition:
['Yes', 'No']
Categories (2, object): ['No', 'Yes']

BusinessTravel:
['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
Categories (3, object): ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']

Department:
['Sales', 'Research & Development', 'Human Resources']
Categories (3, object): ['Human Resources', 'Research & Development', 'Sales']

EducationField:
['Life Sciences', 'Other', 'Medical', 'Marketing', 'Technical Degree', 'Human Resources']
Categories (6, object): ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other',
                         'Technical Degree']

Gender:
['Female', 'Male']
Categories (2, object): ['Female', 'Male']

JobRole:
['Sales Executive', 'Research Scientist', 'Laboratory Technician', 'Manufacturing Director', 'Healthcare Representative', 'Manager', 'Sales Representative', 'Research Director', 'Human Resources']
Categories (9, object): ['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', ...,
    

Validate Numerical Values

In [ ]:
numeric_columns = clean_df.select_dtypes(
    include=np.number
).columns

negative_values = {}

for column in numeric_columns:
    count = (clean_df[column] < 0).sum()

    if count > 0:
        negative_values[column] = count

negative_values

{}

Validate Age

In [ ]:
clean_df["Age"].describe()

,Age
count,1470.000000
mean,36.923810
std,9.135373
min,18.000000
25%,30.000000
50%,36.000000
75%,43.000000
max,60.000000


Validate Distance From Home

In [ ]:
clean_df["DistanceFromHome"].describe()

,DistanceFromHome
count,1470.000000
mean,9.192517
std,8.106864
min,1.000000
25%,2.000000
50%,7.000000
75%,14.000000
max,29.000000


Validate Income

In [ ]:
clean_df["MonthlyIncome"].describe()

,MonthlyIncome
count,1470.000000
mean,6502.931293
std,4707.956783
min,1009.000000
25%,2911.000000
50%,4919.000000
75%,8379.000000
max,19999.000000


Outlier Detection

In [ ]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]

    return outliers, lower_bound, upper_bound

Run Outlier Analysis

In [ ]:
outlier_summary = []

for column in numeric_columns:

    outliers, lower, upper = detect_outliers_iqr(
        clean_df,
        column
    )

    outlier_summary.append({
        "Column": column,
        "Outlier_Count": len(outliers),
        "Lower_Bound": lower,
        "Upper_Bound": upper
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary.sort_values(
    "Outlier_Count",
    ascending=False
)

,Column,Outlier_Count,Lower_Bound,Upper_Bound
20,TrainingTimesLastYear,238,0.500,4.500
15,PerformanceRating,226,3.000,3.000
11,MonthlyIncome,114,-5291.000,16581.000
24,YearsSinceLastPromotion,107,-4.500,7.500
22,YearsAtCompany,104,-6.000,18.000
18,StockOptionLevel,85,-1.500,2.500
19,TotalWorkingYears,63,-7.500,28.500
13,NumCompaniesWorked,52,-3.500,8.500
23,YearsInCurrentRole,21,-5.500,14.500
25,YearsWithCurrManager,14,-5.500,14.500


Logical Consistency Checks

In [ ]:
(
    clean_df["YearsAtCompany"] >
    clean_df["TotalWorkingYears"]
).sum()

np.int64(0)

In [ ]:
(
    clean_df["YearsInCurrentRole"] >
    clean_df["YearsAtCompany"]
).sum()

np.int64(0)

In [ ]:
(
    clean_df["YearsWithCurrManager"] >
    clean_df["YearsAtCompany"]
).sum()

np.int64(0)

In [ ]:
(
    clean_df["YearsSinceLastPromotion"] >
    clean_df["YearsAtCompany"]
).sum()

np.int64(0)

In [ ]:
constant_columns = [
    column
    for column in clean_df.columns
    if clean_df[column].nunique() == 1
]

constant_columns

['EmployeeCount', 'Over18', 'StandardHours']

Remove Constant Columns

In [ ]:
clean_df = clean_df.drop(
    columns=constant_columns
)

In [ ]:
clean_df.shape

(1470, 32)

Feature Engineering

Feature 1 — Age Group

In [ ]:
clean_df["AgeGroup"] = pd.cut(
    clean_df["Age"],
    bins=[17, 25, 35, 45, 55, 100],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56+"
    ]
)

In [ ]:
clean_df["AgeGroup"].value_counts().sort_index()

,count
AgeGroup,
18-25,123
26-35,606
36-45,468
46-55,226
56+,47


Feature 2 — Tenure Group

In [ ]:
clean_df["TenureGroup"] = pd.cut(
    clean_df["YearsAtCompany"],
    bins=[-1, 2, 5, 10, 20, 100],
    labels=[
        "0-2 Years",
        "3-5 Years",
        "6-10 Years",
        "11-20 Years",
        "20+ Years"
    ]
)
clean_df["TenureGroup"].value_counts().sort_index()

,count
TenureGroup,
0-2 Years,342
3-5 Years,434
6-10 Years,448
11-20 Years,180
20+ Years,66


Feature 3 — Experience Group


In [ ]:
clean_df["ExperienceGroup"] = pd.cut(
    clean_df["TotalWorkingYears"],
    bins=[-1, 2, 5, 10, 20, 100],
    labels=[
        "0-2 Years",
        "3-5 Years",
        "6-10 Years",
        "11-20 Years",
        "20+ Years"
    ]
)

Feature 4 — Income Group

In [ ]:
clean_df["IncomeGroup"] = pd.qcut(
    clean_df["MonthlyIncome"],
    q=4,
    labels=[
        "Low",
        "Lower-Middle",
        "Upper-Middle",
        "High"
    ]
)
clean_df["IncomeGroup"].value_counts()

,count
IncomeGroup,
Low,369
High,368
Upper-Middle,367
Lower-Middle,366


Feature 5 — Distance Group

In [ ]:
clean_df["DistanceGroup"] = pd.cut(
    clean_df["DistanceFromHome"],
    bins=[0, 5, 10, 20, 100],
    labels=[
        "0-5 Miles",
        "6-10 Miles",
        "11-20 Miles",
        "20+ Miles"
    ]
)

Feature 6 — Promotion Group

In [ ]:
clean_df["PromotionGroup"] = pd.cut(
    clean_df["YearsSinceLastPromotion"],
    bins=[-1, 2, 5, 10, 100],
    labels=[
        "0-2 Years",
        "3-5 Years",
        "6-10 Years",
        "10+ Years"
    ]
)

Feature 7 — Satisfaction Labels

In [ ]:
satisfaction_map = {
    1: "Low",
    2: "Medium",
    3: "High",
    4: "Very High"
}
clean_df["JobSatisfactionLabel"] = (
    clean_df["JobSatisfaction"]
    .map(satisfaction_map)
)
clean_df["EnvironmentSatisfactionLabel"] = (
    clean_df["EnvironmentSatisfaction"]
    .map(satisfaction_map)
)

clean_df["RelationshipSatisfactionLabel"] = (
    clean_df["RelationshipSatisfaction"]
    .map(satisfaction_map)
)

Work-Life Balance Labels

In [ ]:
worklife_map = {
    1: "Low",
    2: "Medium",
    3: "High",
    4: "Very High"
}

clean_df["WorkLifeBalanceLabel"] = (
    clean_df["WorkLifeBalance"]
    .map(worklife_map)
)

Job Involvement Labels

In [ ]:
clean_df["JobInvolvementLabel"] = (
    clean_df["JobInvolvement"]
    .map(worklife_map)
)

Performance Rating Label

In [ ]:
performance_map = {
    1: "Low",
    2: "High",
    3: "Excellent",
    4: "Outstanding"
}

clean_df["PerformanceRatingLabel"] = (
    clean_df["PerformanceRating"]
    .map(performance_map)
)

Verify New Columns

In [ ]:
clean_df.shape

(1470, 44)

Final Missing-Value Check

In [ ]:
clean_df.isnull().sum()

,0
Age,0
Attrition,0
BusinessTravel,0
DailyRate,0
Department,0
DistanceFromHome,0
Education,0
EducationField,0
EmployeeNumber,0
EnvironmentSatisfaction,0


In [ ]:
clean_df.isnull().sum().sum()

np.int64(0)

Final Duplicate Check

In [ ]:
clean_df.duplicated().sum()

np.int64(0)

In [ ]:
Final Employee ID Check

In [ ]:
clean_df["EmployeeNumber"].nunique()
clean_df["EmployeeNumber"].duplicated().sum()

np.int64(0)

Create Final Data Quality Summary

In [ ]:
final_quality_report = pd.DataFrame({
    "Column": clean_df.columns,
    "Data_Type": clean_df.dtypes.astype(str).values,
    "Missing_Values": clean_df.isnull().sum().values,
    "Unique_Values": clean_df.nunique().values
})

final_quality_report

,Column,Data_Type,Missing_Values,Unique_Values
0,Age,int64,0,43
1,Attrition,category,0,2
2,BusinessTravel,category,0,3
3,DailyRate,int64,0,886
4,Department,category,0,3
5,DistanceFromHome,int64,0,29
6,Education,int64,0,5
7,EducationField,category,0,6
8,EmployeeNumber,int64,0,1470
9,EnvironmentSatisfaction,int64,0,4


Save the Cleaned Dataset

In [ ]:
from pathlib import Path

output_path = Path("../../02_Python/02_Data_Cleaning/HR_Employee_Attrition_Cleaned.csv")

# Create the parent directories if they don't exist
output_path.parent.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(
    output_path,
    index=False
)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.


Better — Use an Explicit Path

In [ ]:
output_path = "../../02_Python/02_Data_Cleaning/HR_Employee_Attrition_Cleaned.csv"

clean_df.to_csv(
    output_path,
    index=False
)

Validate the Exported File

In [ ]:
test_df = pd.read_csv(
    "../../02_Python/02_Data_Cleaning/HR_Employee_Attrition_Cleaned.csv"
)

print("Rows:", test_df.shape[0])
print("Columns:", test_df.shape[1])

Rows: 1470
Columns: 44


In [ ]:
print("Missing:", test_df.isnull().sum().sum())
print("Duplicates:", test_df.duplicated().sum())

Missing: 0
Duplicates: 0


In [ ]:
Before vs After

SyntaxError: invalid syntax (756406334.py, line 1)

In [ ]:
print("BEFORE CLEANING")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nAFTER CLEANING & FEATURE ENGINEERING")
print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])

BEFORE CLEANING
Rows: 1470
Columns: 35

AFTER CLEANING & FEATURE ENGINEERING
Rows: 1470
Columns: 44


### Loading a Sample Dataset (e.g., California Housing from scikit-learn)